In [5]:
# %pip install transformers==5.0.0rc0
# %pip install mistral-common --upgrade
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import Mistral3Model, Mistral3PreTrainedModel, MistralCommonBackend

from dataclasses import dataclass
from typing import Dict, Any, Optional

import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training




In [6]:
import os, yaml
from dotenv import load_dotenv


def load_env():
    load_dotenv()
    return {
        "HF_TOKEN": os.getenv("HF_TOKEN", ""),
        "WANDB_DISABLED": os.getenv("WANDB_DISABLED", "true"),
    }

# ----------------------------
# 1) Config Layer
# ----------------------------

@dataclass
class TrainingConfig:
    CFG = {
        "project": {"seed": 42, "output_dir": "outputs/mistral_multitask"},
        "data": {
            "raw_dir": "data/raw",
            "processed_path": "data/processed/unified.parquet",
            "text_col_candidates": ["text", "joke", "body", "fulltext", "sentence"],
            "cls_label_candidates": ["label", "is_humor", "humor", "funny"],
            "reg_label_candidates": ["score", "rating", "humor_rating", "grade"],
        },
        "labels": {
            "reg_min": 0.0,
            "reg_max": 4.0,
            "cls_from_reg_threshold": 0,
            "reg_quantile": 0.95,
        },
        "model": {"base_id": "mistralai/Ministral-3-8B-Base-2512", "max_length": 128},
        "lora": {
            "enabled": True,
            "r": 16,
            "alpha": 32,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        },
        "train": {
            "task": "multitask",
            "num_train_epochs": 3,
            "learning_rate": 1e-4,
            "per_device_train_batch_size": 4,
            "per_device_eval_batch_size": 8,
            "gradient_accumulation_steps": 1,
            "warmup_steps": 162,
            "weight_decay": 0.0,
            "logging_steps": 10,
            "eval_steps": 300,  
            "save_steps": 300,
            "bf16": True,
            "fp16": False,
        },
        "loss": {"cls_weight": 1.0, "reg_weight": 1.0},
        "cv": {"enabled": False, "folds": 5},
        "hpo": {"enabled": False, "n_trials": 15},
    }
       
    csv_path: str = r"C:\Users\Anwender\humor-project\data\labels\hahackathon_train.csv"

    # column names in CSV
    text_col: str = "text"
    cls_col_raw: str = "is_humor"
    reg_col_raw: str = "humor_rating"

    # masking
    missing_cls_value: int = -1
    missing_reg_value: float = -1.0

    # quant
    use_4bit: bool = False  # set True when bitsandbytes works
 

In [7]:


# ----------------------------
# 2) Data Layer
# ----------------------------

class DataModule:
    def __init__(self, cfg: Dict[str, Any], tcfg: TrainingConfig):
        self.cfg = cfg
        self.tcfg = tcfg

    def load_dataframe(self) -> pd.DataFrame:
        df = pd.read_csv(self.tcfg.csv_path)

        df = df.rename(columns={
            self.tcfg.cls_col_raw: "y_cls",
            self.tcfg.reg_col_raw: "y_reg",
            self.tcfg.text_col: "text",
        })

        df = df.dropna(subset=["text"]).reset_index(drop=True)

        # 1) classification label (keep, but will be overridden for consistency later)
        df["y_cls"] = df["y_cls"].fillna(self.tcfg.missing_cls_value).astype(int)

        # 2) regression: robust numeric conversion
        df["y_reg"] = pd.to_numeric(df["y_reg"], errors="coerce").astype("float32")

        # non-humor => -1
        df.loc[df["y_cls"] == 0, "y_reg"] = self.tcfg.missing_reg_value

        # humor but missing rating => -1
        df.loc[(df["y_cls"] == 1) & (df["y_reg"].isna()), "y_reg"] = self.tcfg.missing_reg_value

        # 3) clip only humor ratings (>=0)
        humor = df["y_reg"] >= 0
        df.loc[humor, "y_reg"] = df.loc[humor, "y_reg"].clip(0.0, 4.0)

        # 4) enforce strict consistency: y_cls derived from y_reg sign
        df["y_cls"] = (df["y_reg"] >= 0).astype(int)

        return df



    @staticmethod
    def tokenize_fn(ex, tokenizer, max_length: int):
        return tokenizer(
            ex["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )

    def build_datasets(self, tokenizer):
        seed = int(self.cfg["project"]["seed"])
        max_len = int(self.cfg["model"]["max_length"])

        df = self.load_dataframe()
        train_df, val_df = train_test_split(df, test_size=0.1, random_state=seed)

        train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
        val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

        train_ds = train_ds.map(lambda x: self.tokenize_fn(x, tokenizer, max_len), batched=True)
        val_ds = val_ds.map(lambda x: self.tokenize_fn(x, tokenizer, max_len), batched=True)

        cols = ["input_ids", "attention_mask", "y_cls", "y_reg"]
        train_ds = train_ds.select_columns(cols)
        val_ds = val_ds.select_columns(cols)

        return train_ds, val_ds



In [8]:
def test_dataset():
    cfg: Dict[str, Any] = {
        "project": {"seed": 42},
        "model": {"max_length": 64},
    }

    tcfg = TrainingConfig(
        csv_path=r"C:\Users\Anwender\humor-project\data\labels\hahackathon_train.csv",
        text_col="text",
        cls_col_raw="is_humor",
        reg_col_raw="humor_rating",
        missing_cls_value=-1,
        missing_reg_value=-1.0,
    )

    dm = DataModule(cfg, tcfg)

    # 1) dataframe
    df = dm.load_dataframe()
    print(df.head(2))
    print("columns:", df.columns.tolist())
    print("y_cls unique sample:", df["y_cls"].unique()[:10])
    print("y_reg min/max:", df["y_reg"].min(), df["y_reg"].max())
    assert "text" in df.columns and "y_cls" in df.columns and "y_reg" in df.columns

    # 2) datasets
    tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    train_ds, val_ds = dm.build_datasets(tok)

    print("train rows:", len(train_ds), "val rows:", len(val_ds))
    sample = train_ds[0]
    print(sample.keys())
    print("input_ids len:", len(sample["input_ids"]))
    assert set(sample.keys()) == {"input_ids", "attention_mask", "y_cls", "y_reg"}
    assert len(sample["input_ids"]) == cfg["model"]["max_length"]
    assert len(sample["attention_mask"]) == cfg["model"]["max_length"]
    
    # property of dataset: rates are only given to those data which are classified as humor  
    bad1 = ((df["y_cls"] == 0) & (df["y_reg"] >= 0)).sum()
    bad2 = ((df["y_cls"] == 1) & (df["y_reg"] < 0)).sum()
    print("check datas with the following two cinditions (cls=0 but reg>=0):", bad1)
    print("check datas with the following two cinditions (cls=1 but reg<0):", bad2)


if __name__ == "__main__":
    test_dataset()

   id                                               text  y_cls  y_reg  \
0   1  TENNESSEE: We're the best state. Nobody even c...      1   2.42   
1   2  A man inserted an advertisement in the classif...      1   2.50   

   humor_controversy  offense_rating  
0                1.0             0.2  
1                1.0             1.1  
columns: ['id', 'text', 'y_cls', 'y_reg', 'humor_controversy', 'offense_rating']
y_cls unique sample: [1 0]
y_reg min/max: -1.0 4.0


Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

train rows: 7200 val rows: 800
dict_keys(['input_ids', 'attention_mask', 'y_cls', 'y_reg'])
input_ids len: 64
check datas with the following two cinditions (cls=0 but reg>=0): 0
check datas with the following two cinditions (cls=1 but reg<0): 0


In [ ]:
# ----------------------------
# 3) Model Layer
# ----------------------------

def _last_token_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    if attention_mask is None:
        return last_hidden_state[:, -1, :]
    lengths = attention_mask.long().sum(dim=1)
    idx = torch.clamp(lengths - 1, min=0)
    batch_idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
    return last_hidden_state[batch_idx, idx, :]

def _get_hidden_size_from_mistral3_config(cfg) -> int:
    if hasattr(cfg, "hidden_size"):
        return int(cfg.hidden_size)

    for attr in ["language_config", "text_config", "language_model_config", "llm_config"]:
        if hasattr(cfg, attr):
            sub = getattr(cfg, attr)
            if hasattr(sub, "hidden_size"):
                return int(sub.hidden_size)

    d = cfg.to_dict() if hasattr(cfg, "to_dict") else dict(cfg)
    for key in ["hidden_size", "language_config", "text_config", "language_model_config", "llm_config"]:
        if key in d and isinstance(d[key], dict) and "hidden_size" in d[key]:
            return int(d[key]["hidden_size"])
        if key == "hidden_size" and key in d:
            return int(d[key])

    raise AttributeError("Could not find hidden_size in Mistral3Config.")


class Mistral3ForHumorMultiTask(Mistral3PreTrainedModel):
    """
    Multitask model:
      - CLS head: humor vs non-humor
      - REG head: humor rating (raw scalar)

    IMPORTANT:
      - forward() returns raw outputs (good for training / computing loss)
      - predict_rule() returns rule-based final outputs (good for inference)
    """
    def __init__(self, config):
        super().__init__(config)

        # your backbone name stays "model"
        self.model = Mistral3Model(config)

        # ✅ ALIAS so checkpoint keys `language_model.*` can load
        # this does NOT duplicate weights, it's the same module object
        self.language_model = self.model

        hidden = _get_hidden_size_from_mistral3_config(config)

        self.dropout = nn.Dropout(getattr(config, "classifier_dropout", 0.1))
        self.cls_head = nn.Linear(hidden, 2)
        self.reg_head = nn.Linear(hidden, 1)

        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, **kwargs):
        out = self.model(
            input_ids=input_ids if inputs_embeds is None else None,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            **kwargs,
        )
        pooled = _last_token_pool(out.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)

        logits_cls = self.cls_head(pooled)              # [B,2]
        pred_reg = self.reg_head(pooled).squeeze(-1)    # [B]

        # Raw outputs for training / metrics
        return {"logits_cls": logits_cls, "pred_reg": pred_reg}

    @torch.inference_mode()
    def predict_rule(
        self,
        input_ids=None,
        attention_mask=None,
        inputs_embeds=None,
        threshold: float = 0.5,
        rating_epsilon: float = 0.05,
        **kwargs
    ):
        """
        Business rule (strict boundary + rating gate):
        1) p_humor = softmax(logits)[humor_class=1]
        2) rating_clamp = clamp(pred_reg, 0, 4)              # keep FLOAT
        3) is_humor = (p_humor > threshold) AND (rating_clamp > rating_epsilon)
        4) final = rating_clamp if humor else -1

        Returns:
        {
            "p_humor": [B] float,
            "is_humor": [B] int 0/1,
            "rating_raw": [B] float,
            "rating_clamp": [B] float in [0,4],
            "final": [B] float (-1.0 if not humor else [0,4])
        }
        """
        out = self.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            **kwargs
        )
        logits_cls = out["logits_cls"]   # [B,2]
        rating_raw = out["pred_reg"]     # [B] float

        probs = F.softmax(logits_cls, dim=-1)   # [B,2]
        p_humor = probs[:, 1]                   # [B]

        # ✅ keep float (NO .long())
        rating_clamp = torch.clamp(rating_raw, 0.0, 4.0)  # [B] float

        # strict boundary: 0.500 -> NOT humor
        is_humor_prob = (p_humor > threshold)

        # rating gate: if rating is basically 0 => NOT humor
        is_humor = (is_humor_prob & (rating_clamp > rating_epsilon)).long()

        # ✅ keep final float: -1.0 or [0,4]
        final = torch.where(
            is_humor == 1,
            rating_clamp,
            torch.full_like(rating_clamp, -1.0),
        )

        return {
            "p_humor": p_humor,
            "is_humor": is_humor,
            "rating_raw": rating_raw,
            "rating_clamp": rating_clamp,
            "final": final,
        }


In [ ]:
from typing import Dict, Any, Optional
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import MistralCommonBackend

# your class
# from your_model_file import Mistral3ForHumorMultiTask


class ModelFactory:
    def __init__(self, cfg: Dict[str, Any], env: Dict[str, str], tcfg: "TrainingConfig"):
        self.cfg = cfg
        self.env = env
        self.tcfg = tcfg

    def _build_bnb(self) -> Optional[BitsAndBytesConfig]:
        if not self.tcfg.use_4bit:
            return None
        # for your config we keep compute dtype = fp16 (bnb stable default)
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

    def load_tokenizer(self):
        model_id = self.cfg["model"]["base_id"]
        tok = MistralCommonBackend.from_pretrained(model_id)
        return tok

    def load_model(self):
        model_id = self.cfg["model"]["base_id"]
        bnb_cfg = self._build_bnb()

        # config says bf16=True, fp16=False
        dtype = torch.bfloat16

        load_kwargs = dict(
            token=self.env.get("HF_TOKEN", None),
            trust_remote_code=True,
            torch_dtype=dtype,
            device_map="auto",
        )
        if bnb_cfg is not None:
            load_kwargs["quantization_config"] = bnb_cfg

        model = Mistral3ForHumorMultiTask.from_pretrained(model_id, **load_kwargs)

        # speed: disable KV cache in training
        if hasattr(model, "config") and hasattr(model.config, "use_cache"):
            model.config.use_cache = False

        # if 4bit -> kbit training prep
        if bnb_cfg is not None:
            model = prepare_model_for_kbit_training(model)

        # LoRA (cfg)
        if bool(self.cfg["lora"]["enabled"]):
            lora_cfg = LoraConfig(
                r=int(self.cfg["lora"]["r"]),
                lora_alpha=int(self.cfg["lora"]["alpha"]),
                lora_dropout=float(self.cfg["lora"]["dropout"]),
                target_modules=list(self.cfg["lora"]["target_modules"]),  # EXACT from cfg
                bias="none",
                modules_to_save=["cls_head", "reg_head"],  # IMPORTANT
                task_type="FEATURE_EXTRACTION",
            )
            model = get_peft_model(model, lora_cfg)

        return model


In [11]:
print("\n==============================")
print("✅ Running ModelFactory test ")
print("==============================")

tcfg = TrainingConfig()
cfg = tcfg.CFG
env = load_env()

factory = ModelFactory(cfg=cfg, env=env, tcfg=tcfg)
tok = factory.load_tokenizer()
model = factory.load_model()

texts = [
    "Why did the scarecrow win an award? Because he was outstanding in his field.",
    "The capital of France is Paris."
]

batch = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=cfg["model"]["max_length"])
# device_map="auto": move inputs to first param device (robust)
device = next(model.parameters()).device
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    out = model(**batch)

print("out keys:", out.keys())
print("logits_cls:", out["logits_cls"].shape, out["logits_cls"].dtype, out["logits_cls"].device)
print("pred_reg  :", out["pred_reg"].shape, out["pred_reg"].dtype, out["pred_reg"].device)

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameter



✅ Running ModelFactory test 


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/222 [00:00<?, ?it/s]

Mistral3ForHumorMultiTask LOAD REPORT from: mistralai/Ministral-3-8B-Base-2512
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
language_model.model.layers.{0...33}.self_attn.o_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.mlp.gate_proj.weight            | UNEXPECTED | 
language_model.model.layers.{0...33}.self_attn.q_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.post_attention_layernorm.weight | UNEXPECTED | 
language_model.model.layers.{0...33}.mlp.down_proj.weight            | UNEXPECTED | 
language_model.model.layers.{0...33}.self_attn.k_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.mlp.up_proj.weight              | UNEXPECTED | 
language_model.model.layers.{0...33}.self_attn.v_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.input_layernorm.weight          | 

out keys: dict_keys(['logits_cls', 'pred_reg'])
logits_cls: torch.Size([2, 2]) torch.bfloat16 cuda:0
pred_reg  : torch.Size([2]) torch.bfloat16 cuda:0


AttributeError: 'Mistral3ForHumorMultiTask' object has no attribute 'print_trainable_parameter'

In [ ]:
from transformers import Trainer

# class MultiTaskTrainer(Trainer):
    def __init__(self, *args, cls_weight=1.0, reg_weight=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.cls_weight = float(cls_weight)
        self.reg_weight = float(reg_weight)
        self.ce = torch.nn.CrossEntropyLoss(reduction="none")
        self.mse = torch.nn.MSELoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # ---- 1) pull labels safely ----
        y_cls = inputs.pop("y_cls", None)
        y_reg = inputs.pop("y_reg", None)

        # If labels are missing entirely, just do forward (some Trainer modes can do this)
        outputs = model(**inputs)
        logits_cls = outputs["logits_cls"]  # [B,2]
        pred_reg = outputs["pred_reg"]      # [B]

        device = logits_cls.device
        loss = torch.tensor(0.0, device=device, dtype=torch.float32)

        # ---- 2) sanitize model outputs (prevents NaN loss explosion) ----
        # Note: this does not fix training instability, but prevents immediate crash.
        logits_cls = torch.nan_to_num(logits_cls, nan=0.0, posinf=50.0, neginf=-50.0)
        pred_reg = torch.nan_to_num(pred_reg, nan=0.0, posinf=4.0,  neginf=0.0)

        # ---- 3) classification loss ----
        if y_cls is not None:
            if not torch.is_tensor(y_cls):
                y_cls = torch.tensor(y_cls)
            y_cls = y_cls.to(device=device)

            # labels must be long for CrossEntropy
            y_cls = y_cls.long()

            # missing label sentinel
            cls_mask = (y_cls != -1)

            if cls_mask.any():
                cls_loss_vec = self.ce(logits_cls[cls_mask], y_cls[cls_mask])
                cls_loss = cls_loss_vec.mean().float()
                loss = loss + self.cls_weight * cls_loss

        # ---- 4) regression loss ----
        if y_reg is not None:
            if not torch.is_tensor(y_reg):
                y_reg = torch.tensor(y_reg)
            y_reg = y_reg.to(device=device)

            # regression in float32 for stability (even if model is bf16)
            y_reg_f = y_reg.float()
            pr_f = pred_reg.float().view(-1)

            # robust mask: exclude missing sentinel AND NaNs/Infs
            reg_mask = (y_reg_f != -1.0) & torch.isfinite(y_reg_f)

            if reg_mask.any():
                yr = y_reg_f[reg_mask].view(-1)
                pr = pr_f[reg_mask].view(-1)

                # optional: clamp predictions to training range (helps stability)
                pr = torch.clamp(pr, 0.0, 4.0)

                reg_loss_vec = self.mse(pr, yr)
                reg_loss = reg_loss_vec.mean().float()
                loss = loss + self.reg_weight * reg_loss

        return (loss, outputs) if return_outputs else loss


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def _sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def compute_metrics(eval_pred):
    """
    Robust multi-task metrics for:
      - y_cls: int {0,1}, missing = -1
      - y_reg: float, missing = -1.0

    Handles:
      - predictions as (cls_logits, reg_pred) tuple/list OR dict
      - labels as (y_cls, y_reg) tuple/list OR (N,2) array OR dict
      - NaN/Inf in preds or labels (won't crash)
      - empty masks (returns nan metrics)
    """

    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    # --------------------
    # 1) Unpack labels safely
    # --------------------
    y_cls = None
    y_reg = None

    if labels is None:
        # No labels -> can't compute anything
        return {
            "cls_acc": np.nan, "cls_precision": np.nan, "cls_recall": np.nan, "cls_f1": np.nan,
            "reg_mae": np.nan, "reg_rmse": np.nan, "reg_r2": np.nan,
            "combined_score": np.nan, "reg_good_exp_neg_rmse": np.nan,
        }

    if isinstance(labels, (tuple, list)) and len(labels) >= 2:
        y_cls = np.asarray(labels[0])
        y_reg = np.asarray(labels[1])
    elif isinstance(labels, dict):
        y_cls = np.asarray(labels.get("y_cls"))
        y_reg = np.asarray(labels.get("y_reg"))
    else:
        labels = np.asarray(labels)
        if labels.ndim == 2 and labels.shape[1] >= 2:
            y_cls = labels[:, 0]
            y_reg = labels[:, 1]
        else:
            raise ValueError(f"Labels format not understood. Got shape {labels.shape}")

    # normalize shapes
    y_cls = np.asarray(y_cls).reshape(-1)
    y_reg = np.asarray(y_reg).reshape(-1)

    # --------------------
    # 2) Unpack predictions safely
    # --------------------
    cls_logits = None
    reg_pred = None

    if preds is None:
        return {
            "cls_acc": np.nan, "cls_precision": np.nan, "cls_recall": np.nan, "cls_f1": np.nan,
            "reg_mae": np.nan, "reg_rmse": np.nan, "reg_r2": np.nan,
            "combined_score": np.nan, "reg_good_exp_neg_rmse": np.nan,
        }

    if isinstance(preds, (tuple, list)) and len(preds) >= 2:
        cls_logits = np.asarray(preds[0])
        reg_pred   = np.asarray(preds[1])
    elif isinstance(preds, dict):
        cls_logits = np.asarray(preds.get("cls_logits", preds.get("logits_cls")))
        reg_pred   = np.asarray(preds.get("reg_pred", preds.get("pred_reg", preds.get("logits_reg"))))
    else:
        raise ValueError(
            "Predictions format not understood. "
            "Make your trainer return (cls_logits, reg_pred) during eval."
        )

    # --------------------
    # 3) Sanitize NaN/Inf in predictions (prevents sklearn crash)
    # --------------------
    cls_logits = np.nan_to_num(cls_logits, nan=0.0, posinf=50.0, neginf=-50.0)
    reg_pred   = np.nan_to_num(reg_pred,   nan=0.0, posinf=4.0,  neginf=0.0)

    # Regression predictions ensure 1D
    reg_pred = np.asarray(reg_pred).reshape(-1)

    # Optional: clip reg predictions to label range
    reg_pred = np.clip(reg_pred, 0.0, 4.0)

    # --------------------
    # 4) Convert logits -> p_pos -> predicted class
    # --------------------
    if cls_logits.ndim == 2 and cls_logits.shape[1] == 2:
        z = cls_logits - cls_logits.max(axis=1, keepdims=True)
        p = np.exp(z)
        denom = p.sum(axis=1, keepdims=True)
        denom = np.clip(denom, 1e-12, None)  # avoid divide-by-zero
        p = p / denom
        p_pos = p[:, 1]
    else:
        cls_logits = cls_logits.reshape(-1)
        p_pos = _sigmoid(cls_logits)

    yhat_cls = (p_pos >= 0.5).astype(int)

    metrics = {}

    # =====================
    # 5) Classification metrics (ignore y_cls == -1 and NaN/Inf)
    # =====================
    y_cls_num = np.nan_to_num(y_cls, nan=-1.0)
    cls_mask = (y_cls_num != -1) & np.isfinite(y_cls_num)

    if np.any(cls_mask):
        y_cls_valid = y_cls_num[cls_mask].astype(int)
        yhat_valid  = yhat_cls[cls_mask]

        acc = accuracy_score(y_cls_valid, yhat_valid)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_cls_valid, yhat_valid, average="binary", zero_division=0
        )

        metrics.update({
            "cls_acc": float(acc),
            "cls_precision": float(prec),
            "cls_recall": float(rec),
            "cls_f1": float(f1),
        })
    else:
        metrics.update({
            "cls_acc": np.nan,
            "cls_precision": np.nan,
            "cls_recall": np.nan,
            "cls_f1": np.nan,
        })

    # =====================
    # 6) Regression metrics (ignore y_reg == -1.0 and NaN/Inf)
    # =====================
    y_reg_num = np.nan_to_num(y_reg, nan=-1.0)

    reg_mask = (y_reg_num != -1.0) & np.isfinite(y_reg_num) & np.isfinite(reg_pred)

    if np.any(reg_mask):
        y_reg_valid = y_reg_num[reg_mask].astype(float)
        ypred_valid = reg_pred[reg_mask].astype(float)

        mae = mean_absolute_error(y_reg_valid, ypred_valid)
        rmse = np.sqrt(mean_squared_error(y_reg_valid, ypred_valid))

        # r2_score can still be weird if y is constant; catch that safely
        try:
            r2 = r2_score(y_reg_valid, ypred_valid)
        except Exception:
            r2 = np.nan

        metrics.update({
            "reg_mae": float(mae),
            "reg_rmse": float(rmse),
            "reg_r2": float(r2),
        })
    else:
        metrics.update({
            "reg_mae": np.nan,
            "reg_rmse": np.nan,
            "reg_r2": np.nan,
        })

    # =====================
    # 7) Combined metric
    # =====================
    cls_f1 = metrics.get("cls_f1", np.nan)
    reg_rmse = metrics.get("reg_rmse", np.nan)

    if np.isfinite(cls_f1) and np.isfinite(reg_rmse):
        reg_good = float(np.exp(-reg_rmse))
        eps = 1e-12
        combined = 2.0 * (cls_f1 * reg_good) / (cls_f1 + reg_good + eps)
        metrics["combined_score"] = float(combined)
        metrics["reg_good_exp_neg_rmse"] = float(reg_good)
    else:
        metrics["combined_score"] = np.nan
        metrics["reg_good_exp_neg_rmse"] = np.nan

    return metrics


In [ ]:


# ----------------------------
# 5) Run Layer
# ----------------------------

def build_training_args(cfg: Dict[str, Any]) -> TrainingArguments:
    return TrainingArguments(
        output_dir=cfg["project"]["output_dir"],

        num_train_epochs=float(cfg["train"]["num_train_epochs"]),
        learning_rate=float(cfg["train"]["learning_rate"]),
        max_grad_norm=1.0,

        per_device_train_batch_size=int(cfg["train"]["per_device_train_batch_size"]),
        per_device_eval_batch_size=int(cfg["train"]["per_device_eval_batch_size"]),
        gradient_accumulation_steps=int(cfg["train"]["gradient_accumulation_steps"]),

        warmup_ratio=float(cfg["train"]["warmup_ratio"]),
        weight_decay=float(cfg["train"]["weight_decay"]),

        # Logging and saving
        logging_steps=int(cfg["train"]["logging_steps"]),
        logging_strategy="steps",
        eval_strategy="epoch",
        save_strategy="epoch",
        log_level="info",
        disable_tqdm=False,   # progress bar
        # still must be ints if present
        eval_steps=int(cfg["train"]["eval_steps"]),
        save_steps=int(cfg["train"]["save_steps"]),

        fp16=bool(cfg["train"]["fp16"]),
        bf16=bool(cfg["train"]["bf16"]),

        report_to = "none",
        logging_first_step = True, 
        remove_unused_columns=False,

        load_best_model_at_end=True,
        metric_for_best_model="combined_score",
        greater_is_better=True,
    )



In [ ]:

def run(tcfg: TrainingConfig, ):
    cfg = tcfg.CFG
    env = load_env()


    print("Loading model ")

    cfg = tcfg.CFG
    env = load_env()
    factory = ModelFactory(cfg=cfg, env=env, tcfg=tcfg)

    # 1) Load tokenizer + model from factory
    tok = factory.load_tokenizer()
    model = factory.load_model()

    # pick a usable device (works with device_map="auto" too)
    device = getattr(model, "device", None)
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    data = DataModule(cfg, tcfg)
    train_ds, val_ds = data.build_datasets(tok)

    args = build_training_args(cfg)

    trainer = MultiTaskTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        cls_weight=cfg["loss"]["cls_weight"],
        reg_weight=cfg["loss"]["reg_weight"],
        compute_metrics=compute_metrics,
    )
    trainer.label_names = ["y_cls", "y_reg"]   

    trainer.train()
    trainer.save_model(cfg["project"]["output_dir"])
    tokenizer.save_pretrained(cfg["project"]["output_dir"]) 


if __name__ == "__main__":
    run(TrainingConfig())


Loading model 


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/222 [00:00<?, ?it/s]

Mistral3ForHumorMultiTask LOAD REPORT from: mistralai/Ministral-3-8B-Base-2512
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
language_model.model.layers.{0...33}.self_attn.o_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.post_attention_layernorm.weight | UNEXPECTED | 
language_model.model.layers.{0...33}.self_attn.q_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.mlp.up_proj.weight              | UNEXPECTED | 
language_model.model.layers.{0...33}.mlp.gate_proj.weight            | UNEXPECTED | 
language_model.model.layers.{0...33}.input_layernorm.weight          | UNEXPECTED | 
language_model.model.layers.{0...33}.self_attn.k_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.self_attn.v_proj.weight         | UNEXPECTED | 
language_model.model.layers.{0...33}.mlp.down_proj.weight            | 

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The model is already on multiple devices. Skipping the move to device specified in `args`.
***** Running training *****
  Num examples = 7,200
  Num Epochs = 3
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 1
  Total optimization steps = 5,400
  Number of trainable parameters = 17,641,475


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:

print("\n==============================")
print("✅ Running ModelFactory test ")
print("==============================")
tcfg = TrainingConfig()
cfg = tcfg.CFG
env = load_env()
factory = ModelFactory(cfg=cfg, env=env, tcfg=tcfg)

# 1) Load tokenizer + model from factory
tok = factory.load_tokenizer()
model = factory.load_model()

# pick a usable device (works with device_map="auto" too)
device = getattr(model, "device", None)
if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Test texts
texts = [
    "Why did the scarecrow win an award? Because he was outstanding in his field.",
    "The capital of France is Paris."
]

# 3) Tokenization
batch = tok(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128,
)
batch = {k: v.to(device) for k, v in batch.items()}

print("\n=== 1) Tokenization ===")
print("num_texts:", len(texts))
print("input_ids shape:", tuple(batch["input_ids"].shape), "(B,T)")
print("attention_mask shape:", tuple(batch["attention_mask"].shape), "(B,T)")

# 4) Predict with integrated rule
threshold = 0.5
rating_epsilon = 0.05

print("\n=== 2) Model predict_rule ===")
print(f"threshold={threshold}  rating_epsilon={rating_epsilon}")

with torch.no_grad():
    pred = model.predict_rule(**batch, threshold=threshold, rating_epsilon=rating_epsilon)

# 5) Print results nicely
p_humor = pred["p_humor"].detach().cpu().tolist()
rating_raw = pred["rating_raw"].detach().cpu().tolist()
rating_clamp = pred["rating_clamp"].detach().cpu().tolist()
is_humor = pred["is_humor"].detach().cpu().tolist()
final = pred["final"].detach().cpu().tolist()

for i, text in enumerate(texts):
    print("\n------------------------------")
    print(f"Text {i}: {text}")
    print(f"p_humor        : {p_humor[i]:.4f}")
    print(f"rating_raw     : {rating_raw[i]:.4f}")
    print(f"rating_clamp : {rating_clamp[i]}   (0..4)")
    print(f"is_humor       : {int(is_humor[i])}       (1=humor, 0=not humor)")
    print(f"FINAL rating         : {int(final[i])}       (-1 if not humor else 0..4)")

print("\n✅ Done.\n")
